In [0]:
%pip install datasets

In [0]:
pip install python-dotenv huggingface_hub

In [0]:
from dotenv import load_dotenv
import os

load_dotenv() 

hf_token = os.getenv("HF_TOKEN")
print(hf_token[:10])

In [0]:
ggdds = load_dataset(
    "leonardoblas/us_election_2024_telegram_distilled",
    streaming=True
)

print(next(iter(ds["train"])))

In [0]:
from collections import Counter
counter = Counter()

for sample in ds["train"]:
    channel = sample.get("channel", "unknown")
    counter[channel] += 1

# in kết quả
for k, v in counter.items():
    print(k, v)

In [0]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="leonardoblas/us_election_2024_telegram_distilled",
    repo_type="dataset"
)

# lọc file .db theo folder
from collections import defaultdict

counter = defaultdict(int)
sum = 0
for f in files:
    if f.endswith(".db"):
        folder = f.split("/")[0]  # channels_x
        counter[folder] += 1
        sum += 1

for k, v in sorted(counter.items()):
    print(k, v)

print(sum)

In [0]:
files

In [0]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id="leonardoblas/us_election_2024_telegram_distilled",
    repo_type="dataset",
    allow_patterns="channels_10/*.db"  # chỉ lấy folder này
)

print(local_path)

In [0]:
from pathlib import Path
import sqlite3

channel_path = Path(local_path) / "channels_10"

for db_file in channel_path.glob("*.db"):
    print(f"\n=== {db_file.name} ===")
    
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    # xem bảng
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    print("Tables:", tables)
    
    # nếu có bảng messages thì đọc
    if "messages" in tables:
        cursor.execute("SELECT COUNT(*) FROM messages;")
        count = cursor.fetchone()[0]
        print("Messages:", count)
        
        # đọc thử 5 dòng
        cursor.execute("SELECT * FROM messages LIMIT 5;")
        rows = cursor.fetchall()
        print("Sample:", rows[:2])
    
    conn.close()